# 02 진단 데이터 탐색

신용 셀프 진단(CREDIT-01) 학습 전에 학원 합성 데이터와 Lending Club 2015를 확인하고, 학습에 쓸 값과 근거를 정리한다.

- 근거: 데이터 처리 기획서 3-9 (문항과 학습 칸 대응, 정답식), 3-10 1~2번, 6-2 / 신용진단_학습데이터_조사 8-1~8-3, 9, 10
- 2026-11-13 결정 반영: 문항 10 보기 5개 (`1년 미만 / 1~3년 / 3~5년 / 5~10년 / 10년 이상`), 문항 8 글 '6개월 현금서비스(단기카드대출) 사용 달'
- 데이터: `ml/data/csv/financial_credit_synthetic_155746x207.csv`, `ml/data/csv/loan_data_2015.csv` (git에 안 올림)
- 칸 단위와 뜻: 학원 데이터 칸 사전 `ml/data/csv/column_dictionary_207.csv` (학원 배포 폴더에서 복사, git에 안 올림)
- 출력은 집계 숫자만 (원본 행은 출력하지 않음). 그래도 GitHub에 올리기 전에 **출력 지우기** (be_common 폴더 구조)
- 맨 아래 **6. 탐색 결과 보고서**에 학습에 쓰는 것과 근거를 정리

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "clean.py").exists())
sys.path.insert(0, str(ML_DIR))
CSV_DIR = ML_DIR / "data" / "csv"

from app.credit import OPTIONS, to_answers, to_finance   # 보기 정의와 나누기 (03, 04, /credit/score와 같이 씀)

ACADEMY_CSV = CSV_DIR / "financial_credit_synthetic_155746x207.csv"
LC_CSV = CSV_DIR / "loan_data_2015.csv"
LABEL = "결과값(연체회차)"     # 관측일 뒤 가상 12개월 중 연체가 생긴 월 수 (0~12)

## 0. 데이터 읽기
기획서 3-1: 학원 155,746행 × 207칸 / Lending Club 421,094행 × 74칸

In [2]:
academy = pd.read_csv(ACADEMY_CSV, low_memory=False)
lc_cols = pd.read_csv(LC_CSV, nrows=0).columns
lc = pd.read_csv(LC_CSV, low_memory=False,
                 usecols=["issue_d", "earliest_cr_line", "loan_status", "inq_last_6mths"])
print("학원 데이터:", academy.shape, "/ Lending Club:", (len(lc), len(lc_cols)))

학원 데이터: (155746, 207) / Lending Club: (421094, 74)


## 1. 기본 확인 (기획서 3-10 1번)
빈 값, ID 중복, 원래 정답 비율 (기획서 3-9: 양성 9.85%)

In [3]:
y = academy[LABEL] > 0      # 원래 정답: 12개월 안 연체 1회 이상
print("빈 값 있는 칸:", int(academy.isna().any().sum()), "개")
print("ID(식별구분) 중복:", int(academy["식별구분"].duplicated().sum()), "개")
print("정답 값(연체 월 수) 분포:", academy[LABEL].value_counts().sort_index().to_dict())
print("연체(1회 이상) 비율: %.2f%%" % (y.mean() * 100))
print("KC5000016 = PE0000025 (기획서 3-9: 같은 칸):", bool((academy["KC5000016"] == academy["PE0000025"]).all()))

빈 값 있는 칸: 0 개
ID(식별구분) 중복: 0 개
정답 값(연체 월 수) 분포: {0: 140405, 1: 4902, 2: 5106, 3: 3069, 4: 1442, 5: 575, 6: 196, 7: 42, 8: 9}
연체(1회 이상) 비율: 9.85%
KC5000016 = PE0000025 (기획서 3-9: 같은 칸): True


## 2. 10문항 보기 나누기 (기획서 3-9 대응표, 3-10 2번)
보기 정의와 나누는 함수는 `ml/app/credit.py` (`OPTIONS`, `to_answers`, `to_finance`), 경계값 테스트는 `ml/app/test_credit.py`

학습 입력은 화면 보기와 똑같은 구간이어야 함. 단위는 칸 사전 기준

| 문항 | 칸 (단위) | 나누는 기준 |
| --- | --- | --- |
| 1 | DQ1251001 (건) | 0 / 1 / 2 이상 |
| 2 | KC5000016 (일) | 0 / 1~9 / 10~29 / 30 이상 |
| 3 | BS0000164 (0/1), BS0000772, B32000100 (건) | 하나라도 1 이상이면 있음 |
| 4 | L00000001 (건) | 0 / 1 / 2~3 / 4 이상 |
| 5 | LC0025202, L00060002, L00140002, L00080002, LA0099212 (천원) | 칸별 금액 > 0이면 그 금융권 이용 (여러 개) |
| 6 | L00000002 (**천원**) | 1천만원 = 10,000천원, 1억원 = 100,000천원 |
| 7 | CL0631905 (%), C00000090 (일) | C00000090 = 0이면 카드 없음, 나머지는 30 / 60 / 90 |
| 8 | CF0600611 (개월) | 0 / 1~2 / 3 이상 |
| 9 | L23001911 (건) + C00000093 (**일**) | C00000093이 1~180일이면 6개월 안 카드 발급 1건 (0은 카드 없음) |
| 10 | C00000090, L23003700 (**일**) 중 큰 값 | 365 / 1,095 / 1,825 / 3,650일. 둘 다 0이면 1년 미만 |

In [4]:
answers = to_answers(academy)
q5 = to_finance(academy)
print("보기 나누기 끝:", answers.shape, "/ 문항 5 금융권:", list(q5.columns))

보기 나누기 끝: (155746, 9) / 문항 5 금융권: ['은행', '카드/캐피탈', '저축은행(상호금융 포함)', '대부업']


## 3. 문항별 보기 분포와 연체율 (학원 원래 정답)
로그 오즈 차 = 첫 보기 대비 연체 위험 크기 (0이면 차이 없음). 조사 8-3: 문항 1, 2, 3, 7, 8은 학원 신호 있음, 4, 5, 6, 9, 10은 없음

In [5]:
def logit(p):
    return np.log(p / (1 - p))


def option_table(col):
    s = answers[col].astype(str)
    order = OPTIONS[col]
    t = pd.DataFrame({"사람 비율%": (s.value_counts(normalize=True) * 100).reindex(order).round(1),
                      "연체율%": (y.groupby(s).mean() * 100).reindex(order).round(2)})
    base = t["연체율%"].iloc[0] / 100
    t["로그 오즈 차(첫 보기 = 0)"] = [round(logit(p / 100) - logit(base), 2) for p in t["연체율%"]]
    return t


for col in answers.columns:
    print(f"\n== {col}")
    print(option_table(col).to_string())


== Q1
       사람 비율%   연체율%  로그 오즈 차(첫 보기 = 0)
Q1                                     
없음       83.1   7.81               0.00
1회       12.9  16.49               0.85
2회 이상     4.0  30.90               1.66

== Q2
        사람 비율%   연체율%  로그 오즈 차(첫 보기 = 0)
Q2                                      
없음        82.7   8.07               0.00
10일 미만     1.4  13.74               0.60
10~29일     5.0  14.42               0.65
30일 이상    10.9  20.83               1.10

== Q3
    사람 비율%   연체율%  로그 오즈 차(첫 보기 = 0)
Q3                                  
없음    97.2   9.47               0.00
있음     2.8  23.13               1.06

== Q4
       사람 비율%   연체율%  로그 오즈 차(첫 보기 = 0)
Q4                                     
0건        9.3   8.84               0.00
1건       17.9   9.30               0.06
2~3건     42.6   9.96               0.13
4건 이상    30.2  10.33               0.17

== Q6
       사람 비율%   연체율%  로그 오즈 차(첫 보기 = 0)
Q6                                     
없음        9.3   8.84               0.00
1천 미만    19

In [6]:
# 문항 5 금융권 (여러 개 가능): 이용자와 미이용자 연체율 비교
pd.DataFrame({"이용 비율%": (q5.mean() * 100).round(1),
              "이용자 연체율%": [round(y[q5[c]].mean() * 100, 2) for c in q5],
              "미이용자 연체율%": [round(y[~q5[c]].mean() * 100, 2) for c in q5]})

,이용 비율%,이용자 연체율%,미이용자 연체율%
은행,41.4,10.07,9.70
카드/캐피탈,56.9,10.08,9.55
저축은행(상호금융 포함),45.2,9.95,9.77
대부업,14.6,10.15,9.80


## 4. Lending Club 보정 값 재현 (조사 8-1 기준)
- 'Issued' 제외, 불량 = Charged Off, Default, Late (31-120 days) → 412,634건 중 1.90%
- 문항 9: 6개월 신용 조회 횟수(inq_last_6mths), 기획서 3-9 값 +0.41 / +0.88
- 문항 10: 대출일 − 첫 신용거래일(earliest_cr_line). 새 보기(10년 이상 = 0)로 다시 계산

In [7]:
lc_used = lc[lc["loan_status"] != "Issued"]
bad = lc_used["loan_status"].isin(["Charged Off", "Default", "Late (31-120 days)"])
print("사용 행:", len(lc_used), "/ 불량 비율: %.2f%%" % (bad.mean() * 100))

issue = pd.to_datetime(lc_used["issue_d"], format="%b-%y")
first = pd.to_datetime(lc_used["earliest_cr_line"], format="%b-%y")
first = first.where(first <= issue, first - pd.DateOffset(years=100))     # 두 자리 연도 '65'가 2065로 읽힌 것 되돌림
years = (issue - first).dt.days / 365.25
print("거래 기간 최소 %.1f년 (3년 미만 자료 없음)" % years.min())


def lc_table(groups, order):
    rate = bad.groupby(groups, observed=False).mean().reindex(order)
    return pd.DataFrame({"대출 수": groups.value_counts().reindex(order),
                         "불량률%": (rate * 100).round(2),
                         "로그 오즈 차(첫 칸 = 0)": [round(logit(p) - logit(rate.iloc[0]), 2) for p in rate]})

사용 행: 412634 / 불량 비율: 1.90%
거래 기간 최소 3.1년 (3년 미만 자료 없음)


In [8]:
q9_lc = pd.cut(lc_used["inq_last_6mths"], bins=[-1, 0, 2, np.inf], labels=["0회", "1~2회", "3회 이상"])
lc_table(q9_lc, ["0회", "1~2회", "3회 이상"])

,대출 수,불량률%,로그 오즈 차(첫 칸 = 0)
inq_last_6mths,,,
0회,251536,1.55,0.00
1~2회,144449,2.32,0.41
3회 이상,16649,3.65,0.88


In [9]:
q10_lc = pd.cut(years, bins=[0, 5, 10, np.inf], labels=["5년 미만", "5~10년", "10년 이상"], right=False)
lc_table(q10_lc, ["10년 이상", "5~10년", "5년 미만"])

,대출 수,불량률%,로그 오즈 차(첫 칸 = 0)
10년 이상,346724,1.77,0.00
5~10년,55969,2.58,0.39
5년 미만,9941,2.85,0.49


## 5. 칸 범위와 단위 확인
2번 보기 나누기에 쓴 기준(일, 천원)이 실제 값 범위와 맞는지

In [10]:
history = academy[["C00000090", "L23003700"]].max(axis=1)
print("문항 9 C00000093 범위: %d ~ %d일 / 0(카드 없음) %d명 / 1~180일 %d명" % (
    academy["C00000093"].min(), academy["C00000093"].max(),
    (academy["C00000093"] == 0).sum(), academy["C00000093"].between(1, 180).sum()))
print("문항 10 거래 기간 최대 %d일 / 두 칸 모두 0: %d명 (1년 미만으로)" % (history.max(), (history == 0).sum()))
print("문항 6 1천만원 이상: %.1f%% / 1억원 이상: %.1f%%" % (
    (academy["L00000002"] >= 10000).mean() * 100, (academy["L00000002"] >= 100000).mean() * 100))
even = [q for q, options in OPTIONS.items() if len(options) % 2 == 0]
print("보기 수가 짝수인 문항 ('잘 모르겠어요' 가운데 규칙 필요):", even)

문항 9 C00000093 범위: 0 ~ 7300일 / 0(카드 없음) 14190명 / 1~180일 9000명
문항 10 거래 기간 최대 7300일 / 두 칸 모두 0: 1326명 (1년 미만으로)
문항 6 1천만원 이상: 70.8% / 1억원 이상: 9.6%
보기 수가 짝수인 문항 ('잘 모르겠어요' 가운데 규칙 필요): ['Q2', 'Q3', 'Q4', 'Q6']


## 6. 탐색 결과 보고서 (2026-11-13 실행 기준)

### 6-1. 데이터 확인

| 항목 | 결과 | 기준 |
| --- | --- | --- |
| 학원 데이터 | 155,746행 × 207칸, 빈 값 0, ID 중복 0 | 기획서 3-1과 같음 |
| 원래 정답 | 연체 1회 이상 9.85% (15,341명) | 기획서 3-9 양성 비율과 같음 |
| 대응 칸 | 3-9 대응표 칸 22개 모두 있음, 모두 숫자. KC5000016 = PE0000025 | 기획서 3-9 |
| Lending Club | 421,094행 × 74칸, 사용 412,634행, 불량 1.90% | 조사 8-1과 같음 |

### 6-2. 문항별 신호와 학습(정답식)에 쓰는 값

로그 오즈 차는 첫 보기 대비 가장 큰 값

| 문항 | 학원 데이터 신호 | Lending Club | 정답식에 쓰는 값 | 근거 |
| --- | --- | --- | --- | --- |
| 1 1년 안 30일 이상 연체 | **강함** (2회 이상 +1.66, 연체율 7.8% → 30.9%) | 같은 방향, 약함 (2회 이상 +0.18) | 학원 로지스틱 값 | 기획서 3-9 정답식, 조사 8-3 |
| 2 3개월 안 최장 연체 | **강함** (30일 이상 +1.10) | 같은 방향 (3개월 이내 +0.35) | 학원 로지스틱 값 | 같음 |
| 3 5년 안 채무조정 | **강함** (있음 +1.06, 사람 2.8%) | 약함 | 학원 로지스틱 값 | 같음 |
| 7 카드 한도 사용 비율 | 약함 (90 이상 +0.23) | 같은 방향 (90% 이상 +0.48) | 학원 로지스틱 값 | 같음. 방향이 두 데이터에서 같음 |
| 8 현금서비스 사용 달 | **강함** (3개월 이상 +1.20) | 칸 없음 | 학원 로지스틱 값 | 기획서 3-9, 조사 8-3 |
| 9 6개월 새 대출, 카드 | **없음** (최대 +0.04) | 1~2회 **+0.41** / 3회 이상 **+0.88** (재현 같음) | Lending Club 값 | 기획서 3-9, 조사 9-2 |
| 10 금융거래 기간 | **없음** (최대 +0.07) | 5년 미만 **+0.49** / 5~10년 **+0.39** / 10년 이상 0 | Lending Club 값 (새 보기) | 조사 8-2, 8-4 (5년 뒤에도 위험이 내려감), 2026-11-13 보기 결정 |
| 4 대출 건수 | 약함 (4건 이상 +0.17) | **반대 방향** (-0.13) | 0 | 기획서 3-9 (근거 상충), 조사 8-3 |
| 6 대출 잔액 | 약함 (최대 +0.16) | **반대 방향** (-0.40) | 0 | 같음 |
| 5 이용 금융권 | 거의 없음 (이용자와 미이용자 차이 0.2~0.5%p) | 칸 없음 | 0 | 기획서 3-9 (근거 없음) |

- 문항 4, 6의 작은 신호는 문항끼리 같이 움직여서 생긴 것으로 봄 (연체 신호가 있는 사람이 대출도 많음, 조사 9-2)
- 문항 10은 Lending Club에 3년 미만 자료가 없어(최소 3.1년), 1년 미만, 1~3년, 3~5년이 같은 값(+0.49)

### 6-3. 03 학습으로 넘길 것

- 모델 입력: 점수가 있는 7문항 (1, 2, 3, 7, 8, 9, 10). 화면은 10문항을 그대로 묻고, 4, 5, 6은 상태와 팁에만 씀 (기획서 3-10 5번)
- 정답: 위 6-2 값으로 다시 만든 '12개월 안 연체', 양성 9.85% 유지 (기획서 3-9 정답식, 잡음 표준편차 0.5)
- 보기 나누기: `ml/app/credit.py`의 `to_answers` (02, 03, 04와 /credit/score가 같은 보기 정의를 씀, 기획서 3-10 2번)

### 6-4. 한계

- 학원 데이터는 교육용 합성 데이터이고, Lending Club은 미국의 승인된 대출만 있으며 정답 정의도 다름 (대출 뒤 수개월 안 연체, 12개월 연체 아님). 크기는 참고 수준 (조사 8-4, 9-3)
- 문항 10 '1년 미만'은 2.6%, 문항 3 '있음'은 2.8%로 적음 (보기별 사람 수가 작아 학습 값이 흔들릴 수 있음)
- 문항 5 '은행' 칸(LC0025202)은 은행 **신용대출** 금액만이라, 은행 담보대출만 있는 사람은 은행 이용으로 안 잡힘 (점수 0이라 학습 영향 없음, 팁 문구에만 영향)